<a href="https://colab.research.google.com/github/EriTeo/Hill-Climbing/blob/main/hill_climbing_romania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗺️ Hill Climbing: Mapa de Rumania
Implementación de Hill Climbing para encontrar una ruta en el mapa de Rumania, desde Arad hasta Bucarest.

In [ ]:
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87}
}

# Distancia en línea recta (heurística) hasta Bucarest.
# Hill Climbing necesita una forma de evaluar qué estado es mejor.
heuristic = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 176,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 100,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374
}

In [ ]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = 0 if parent is None else parent.depth + 1

    def __repr__(self):
        return f"<Node {self.state}>"

    def expand(self, problem):
        return [self.child_node(problem, action)
                for action in problem.actions(self.state)]

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        # action es el nombre de la ciudad destino, el costo está en romania[estado_actual][destino]
        step_cost = problem.graph[self.state][action]
        next_node = Node(next_state, self, action, self.path_cost + step_cost)
        return next_node

class RomaniaProblem:
    def __init__(self, initial, goal, graph, heuristic):
        self.initial = initial
        self.goal = goal
        self.graph = graph
        self.heuristic = heuristic

    def actions(self, state):
        # Las acciones posibles son las ciudades conectadas al estado actual
        return list(self.graph[state].keys())

    def result(self, state, action):
        # Al tomar una acción (ir a una ciudad vecina), el nuevo estado es esa ciudad
        return action

    def value(self, state):
        # Hill Climbing tradicional maximiza el valor.
        # Como queremos MINIMIZAR la distancia a la meta, usamos el valor negativo de la heurística.
        # Así, una distancia menor (ej. -100) es "mayor" que una distancia mayor (ej. -300).
        return -self.heuristic[state]

In [7]:
def hill_climbing(problem):
    """
    Algoritmo de búsqueda local Hill Climbing.
    """
    current = Node(problem.initial)
    print(f"Estado inicial: {current.state} (Distancia estimada a meta: {-problem.value(current.state)})")

    while True:
        neighbors = current.expand(problem)

        if not neighbors:
            print(f"\nCima alcanzada (sin vecinos): {current.state}")
            return current

        # Encontramos el vecino con el valor más alto (menor heurística)
        best_neighbor = max(neighbors, key=lambda node: problem.value(node.state))

        # Comparamos el valor del mejor vecino con el actual.
        # Si el mejor vecino no mejora el valor actual, nos detenemos.
        if problem.value(best_neighbor.state) <= problem.value(current.state):
            print(f"\nÓptimo local alcanzado (ningún vecino es mejor). Deteniendo búsqueda en: {current.state}")
            return current

        # Si el vecino es mejor, nos movemos a él
        current = best_neighbor
        print(f"-> Moviendo a: {current.state} (Distancia estimada a meta: {-problem.value(current.state)})")

        if current.state == problem.goal:
            print(f"\n¡Meta alcanzada!: {current.state}")
            return current

In [8]:
# Inicializamos el problema con Arad como inicio y Bucarest como meta
romania_problem = RomaniaProblem('Arad', 'Bucarest', romania, heuristic)

# Ejecutamos el algoritmo
solucion = hill_climbing(romania_problem)

# Extraemos el camino reconstruyendo desde el nodo final hacia sus padres
path = []
node = solucion
while node:
    path.append(node.state)
    node = node.parent

path.reverse()
print("\nRuta final encontrada por Hill Climbing:", " -> ".join(path))
print(f"Costo total de la ruta: {solucion.path_cost}")

Estado inicial: Arad (Distancia estimada a meta: 366)
-> Moviendo a: Sibiu (Distancia estimada a meta: 253)
-> Moviendo a: Fagaras (Distancia estimada a meta: 176)
-> Moviendo a: Bucarest (Distancia estimada a meta: 0)

¡Meta alcanzada!: Bucarest

Ruta final encontrada por Hill Climbing: Arad -> Sibiu -> Fagaras -> Bucarest
Costo total de la ruta: 450
